In [5]:

# =====================================================
# IMPORTS
# =====================================================


import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    classification_report,
    confusion_matrix
)



In [6]:

# =====================================================
# LOAD DATA
# =====================================================
train = pd.read_csv("Train2.csv")
test = pd.read_csv("Test2_NaN.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nLabel counts:")
print(train["label"].value_counts())

print("\nLabel proportions:")
print(train["label"].value_counts(normalize=True))

Train shape: (1821, 146)
Test shape: (1030, 146)

Label counts:
label
0    1086
1     735
Name: count, dtype: int64

Label proportions:
label
0    0.596376
1    0.403624
Name: proportion, dtype: float64


In [7]:

# =====================================================
# TEMPORAL + PERCENTILES + CORE BAND PRUNED FEATURE ENGINEERING FUNCTION
# =====================================================

def create_features_core_8band_no_missing_count(df):

    feature_dict = {}

    bands = [
        "VV",
        "blue",
        "green",
        "red",
        "nir",
        "nira",
        "swir1",
        "swir2"
    ]

    months = list(range(1, 13))

    for band in bands:

        cols = [
            f"{band}_{month:02d}"
            for month in months
            if f"{band}_{month:02d}" in df.columns
        ]

        band_data = df[cols]

        # =====================================================
        # BASIC TEMPORAL FEATURES
        # =====================================================

        feature_dict[f"{band}_mean"] = band_data.mean(axis=1)

        feature_dict[f"{band}_std"] = band_data.std(axis=1)

        feature_dict[f"{band}_min"] = band_data.min(axis=1)

        feature_dict[f"{band}_max"] = band_data.max(axis=1)

        feature_dict[f"{band}_median"] = band_data.median(axis=1)

        feature_dict[f"{band}_range"] = (
            band_data.max(axis=1)
            - band_data.min(axis=1)
        )

        # =====================================================
        # NOTE:
        # We are intentionally NOT adding:
        #
        # feature_dict[f"{band}_missing_count"]
        #
        # because repeated feature-importance checks showed
        # missing-count features were not contributing much.
        # =====================================================

        # =====================================================
        # PERCENTILE FEATURES
        # =====================================================

        feature_dict[f"{band}_p10"] = (
            band_data.quantile(
                0.10,
                axis=1,
                interpolation="linear"
            )
        )

        feature_dict[f"{band}_p25"] = (
            band_data.quantile(
                0.25,
                axis=1,
                interpolation="linear"
            )
        )

    return pd.DataFrame(feature_dict)

In [8]:

# =====================================================
# CREATE FEATURES
# =====================================================

X = create_features_core_8band_no_missing_count(
    train.copy()
)

y = train["label"]

X_test = create_features_core_8band_no_missing_count(
    test.copy()
)

print("Train feature matrix shape:", X.shape)
print("Test feature matrix shape:", X_test.shape)

print("\nTrain/Test feature columns match:")
print(list(X.columns) == list(X_test.columns))

print("\nTotal NaNs in train features:", X.isna().sum().sum())
print("Total NaNs in test features:", X_test.isna().sum().sum())

print("\nFeature names:")
print(X.columns.tolist())




Train feature matrix shape: (1821, 64)
Test feature matrix shape: (1030, 64)

Train/Test feature columns match:
True

Total NaNs in train features: 0
Total NaNs in test features: 0

Feature names:
['VV_mean', 'VV_std', 'VV_min', 'VV_max', 'VV_median', 'VV_range', 'VV_p10', 'VV_p25', 'blue_mean', 'blue_std', 'blue_min', 'blue_max', 'blue_median', 'blue_range', 'blue_p10', 'blue_p25', 'green_mean', 'green_std', 'green_min', 'green_max', 'green_median', 'green_range', 'green_p10', 'green_p25', 'red_mean', 'red_std', 'red_min', 'red_max', 'red_median', 'red_range', 'red_p10', 'red_p25', 'nir_mean', 'nir_std', 'nir_min', 'nir_max', 'nir_median', 'nir_range', 'nir_p10', 'nir_p25', 'nira_mean', 'nira_std', 'nira_min', 'nira_max', 'nira_median', 'nira_range', 'nira_p10', 'nira_p25', 'swir1_mean', 'swir1_std', 'swir1_min', 'swir1_max', 'swir1_median', 'swir1_range', 'swir1_p10', 'swir1_p25', 'swir2_mean', 'swir2_std', 'swir2_min', 'swir2_max', 'swir2_median', 'swir2_range', 'swir2_p10', 'swir2_

In [9]:

# =====================================================
# CATBOOST MODEL
# =====================================================
model = CatBoostClassifier(
    iterations=5000,
    depth=10,
    learning_rate=0.01,
    l2_leaf_reg=10,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200
)

model.fit(
    X,
    y
)


0:	total: 200ms	remaining: 16m 38s
200:	total: 7.56s	remaining: 3m
400:	total: 14.9s	remaining: 2m 50s
600:	total: 22.1s	remaining: 2m 42s
800:	total: 29.3s	remaining: 2m 33s
1000:	total: 36.5s	remaining: 2m 25s
1200:	total: 44s	remaining: 2m 19s
1400:	total: 51s	remaining: 2m 11s
1600:	total: 58.1s	remaining: 2m 3s
1800:	total: 1m 5s	remaining: 1m 55s
2000:	total: 1m 12s	remaining: 1m 48s
2200:	total: 1m 19s	remaining: 1m 41s
2400:	total: 1m 26s	remaining: 1m 33s
2600:	total: 1m 33s	remaining: 1m 26s
2800:	total: 1m 40s	remaining: 1m 18s
3000:	total: 1m 47s	remaining: 1m 11s
3200:	total: 1m 54s	remaining: 1m 4s
3400:	total: 2m 1s	remaining: 57.3s
3600:	total: 2m 8s	remaining: 50.1s
3800:	total: 2m 15s	remaining: 42.9s
4000:	total: 2m 23s	remaining: 35.8s
4200:	total: 2m 30s	remaining: 28.6s
4400:	total: 2m 37s	remaining: 21.5s
4600:	total: 2m 44s	remaining: 14.3s
4800:	total: 2m 52s	remaining: 7.13s
4999:	total: 2m 59s	remaining: 0us


CatBoostClassifier(depth=10, eval_metric='AUC', iterations=5000, l2_leaf_reg=10, learning_rate=0.01, loss_function='Logloss', random_seed=42, verbose=200)

In [10]:

# =====================================================
# FEATURE IMPORTANCE
# =====================================================
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.get_feature_importance()
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 50 Features:")
print(
    importance.head(50)
)

importance.to_csv(
    "FullTrain_Champion8Band_NoMissingCount_Importance.csv",
    index=False
)


Top 50 Features:
         Feature  Importance
34       nir_min    5.538837
42      nira_min    4.187315
2         VV_min    3.259316
7         VV_p25    3.175429
46      nira_p10    2.992015
57     swir2_std    2.986067
50     swir1_min    2.944203
49     swir1_std    2.940151
63     swir2_p25    2.842995
53   swir1_range    2.697001
22     green_p10    2.499327
6         VV_p10    2.369587
14      blue_p10    2.145129
62     swir2_p10    2.105347
20  green_median    1.954935
23     green_p25    1.938137
18     green_min    1.887401
31       red_p25    1.865806
0        VV_mean    1.839961
38       nir_p10    1.778060
54     swir1_p10    1.739889
61   swir2_range    1.723677
56    swir2_mean    1.711818
55     swir1_p25    1.674628
40     nira_mean    1.653987
16    green_mean    1.641089
43      nira_max    1.590386
3         VV_max    1.560723
30       red_p10    1.486617
8      blue_mean    1.454133
58     swir2_min    1.311843
45    nira_range    1.232453
10      blue_min    1.208

In [11]:
#GENERATE PREDICTIONS
test_prob_matrix = model.predict_proba(
    X_test
)

print("Model classes:", model.classes_)

class_1_index = list(model.classes_).index(1)

prob_1 = test_prob_matrix[:, class_1_index]

prob_0 = 1 - prob_1

test_preds = (
    prob_1 >= 0.5
).astype(int)

confidence = np.maximum(
    prob_0,
    prob_1
)

Model classes: [0 1]


In [12]:
#SAVE RESULTS
submission = pd.DataFrame({

    "ID": test["ID"],

    "TargetF1": test_preds,

    "TargetRAUC": prob_1
})

submission.to_csv(
    "Experiment_FullTrain_Champion8Band_NoMissingCount_CatBoost_submit.csv",
    index=False
)

print(submission.head())


                   ID  TargetF1  TargetRAUC
0  ID_TS_NEW_SBZAYD5I         1    0.968906
1  ID_TS_NEW_7SPRN3PB         1    0.999581
2  ID_TS_NEW_LZOWPHLC         1    0.952814
3  ID_TS_NEW_DN6TUF64         1    0.919680
4  ID_TS_NEW_95N82M49         0    0.000589
